# Prompt Engineering

### Controlling the model's generative capabilities

<img src="./images/genai_81.png" width=1000 />

<img src="./images/genai_82.png" width=1000 />

<img src="./images/genai_83.png" width=1000 />

top_p , also known as nucleus sampling, is a sampling technique that controls which subset of tokens (the nucleus) the LLM can consider. 

It will consider tokens until it reaches their cumulative probability.

If we set top_p to 0.1, it will consider tokens until it reaches that value.

If we set top_p to 1, it will consider all tokens.

top_k parameter controls exactly how many tokens the LLM can consider. If you set it to 100, LLM will consider only the top 100 most probable tokens.

### Code Demo

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import os

device = "mps" if torch.backends.mps.is_available() else "cpu"

https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard#/

In [ ]:
def load_model(model, gated_model=False):
    if gated_model:
        os.environ["HF_TOKEN"] = ""
        tokenizer = AutoTokenizer.from_pretrained(model, use_auth_token=True)
        model = AutoModelForCausalLM.from_pretrained(
            model,
            torch_dtype=torch.float16,
            use_auth_token=True
        ).to(device)

    else:
        tokenizer = AutoTokenizer.from_pretrained(model)

        model = AutoModelForCausalLM.from_pretrained(
            model,
            torch_dtype=torch.float16,
        ).to(device)

    return tokenizer, model

In [ ]:
def generate_response(tokenizer, model, prompt, max_new_tokens, temperature, top_p):
    messages = [
        {"role": "user", "content": prompt}
    ]
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
        )
    generated_ids = outputs[0, input_ids.shape[-1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return response

In [ ]:
tokenizer, model = load_model("microsoft/Phi-3-mini-4k-instruct")

In [ ]:
# using model from gated repo requires Hugging face token - 
# https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
# https://huggingface.co/settings/gated-repos

tokenizer, model = load_model("meta-llama/Llama-3.1-8B-Instruct", gated_model=True)

<img src="./images/genai_85.png" width=1000 />

In [ ]:
tokenizer, model = load_model("Qwen/Qwen2.5-7B-Instruct")

<img src="./images/genai_84.png" width=1000 />

In [ ]:
prompt_structure = {
    'persona': """You are a mathematical genius who is an expert in solving math and related engineering problems.
""",
    'context': """Data contains a comma separated list of numbers spread across different lines. Each line may contain positive and negative numbers. Understand that any number which has a hyphen as a prefix is a negative number.
""",
    'data': """1,2,3,-1,-1
-1,2,3,-2,-1
-1,2,-3""",
    'task': 'For each line in the input data, count the negative number. Store the count in your memory for each line. Think step-by-step.',
    'examples': """Input :
    -1,-2,3,-1,-1,
    -4,-5,0

    Output:
    {
    "Line 1" : 4,
    "Line 2" : 2
    }
    """,
    'output_format': """Provide it in JSON format""",
    'guardrails' : """Strictly adhere to the output format. 
    Use the examples given as only a guideline and do not copy the output.
    Do not clutter the output with additional text like the input code that you are generating.
Verify the calculation twice.""",
    
}

In [ ]:
prompt_structure

In [ ]:
def get_prompt(prompt_structure):
    prompt = ""
    for key, value in prompt_structure.items():
        prompt = prompt + key + ":" + value + '\n'
    return prompt

In [ ]:
model

In [ ]:
print(get_prompt(prompt_structure=prompt_structure))

In [ ]:
response = generate_response(
    tokenizer=tokenizer, 
    model=model,
    prompt=get_prompt(prompt_structure=prompt_structure),
    max_new_tokens=2000,
    temperature=0.1,
    top_p=0.1
)
print(response)

##### Using quantized model

Quantization reduces the number of bits required to represent the
parameters of an LLM while attempting to maintain most of the original
information. 

This comes with some loss in precision but often makes up for
it as the model is much faster to run, requires less VRAM, and is often
almost as accurate as the original.

<img src="./images/genai_86.png" width=1000 />

llama_cpp is a Python binding over the llama.cpp C++ inference engine.

It allows you to run GGUF models locally.

This is NOT PyTorch or TensorFlow inference.

It is low-level, highly optimized C++ inference.

GGUF = Generalized GGML Unified Format

q5_0 - quantization level (q4 / q8) - The model is sharded into 2 files

n_gpu_layers - All possible layers on GPU

n_threads=8 - Number of CPU threads used.

max_tokens = The model will generate up to 100 new tokens

n_ctx is not just your prompt. It includes everything :

<pre>
[ System instructions ]
[ Context / documents ]
[ User prompt ]
[ Model’s generated output ]


n_ctx = 8192

Prompt = 6000 tokens
max_tokens = 1000
--------------------------------
Total = 7000 tokens  ✅ OK

Prompt = 8000 tokens
max_tokens = 1000
--------------------------------
Total = 9000 tokens ❌ OVERFLOW

</pre>

In [ ]:
# https://huggingface.co/Qwen/Qwen2.5-7B-Instruct-GGUF/blob/main/README.md

from llama_cpp import Llama

model_path = "/Users/manojkumar_rajendran/qwen2.5-7b-instruct-q5_0-00001-of-00002.gguf"

llm = Llama(
    model_path=model_path,
    n_ctx=8192,
    n_gpu_layers=-1,
    n_threads=8,
    verbose=False,
)

out = llm(
    get_prompt(prompt_structure=prompt_structure), 
    max_tokens=100, 
    temperature=0.1,
    top_p=0.1
)
print(out["choices"][0]["text"])

### Comparing SLM with LLM

In [ ]:
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

df = pd.read_csv("images/Phi3mini_vs_Llama31_8B_comprehensive_comparison.csv")

df.style.set_properties(**{
    "white-space": "pre-wrap",
    "text-align": "left"
})

### Best practices for prompt

Specificity - Accurately describe what you want to achieve

Hallucination - LLMs generate incorrect information confidently. "I don't know" can be added in the prompt.

Order - Instruction should be at beginning (primacy effect) or the end (recency effect) especially for long prompts. Info in the middle section is mostly forgotten.

##### Mandatory components for a good prompt

Persona

Instruction

Context

Format

Audience

Tone

Data

### Advanced techniques for prompt

In-context learning (zero-shot or few-shot)

Chain prompting (multiple prompts & LLM calls)

Chain-of-thought (Think step-by-step)

Self-consistency (result from majority voting - multiple LLM calls)

Tree-of-thought

<pre>
    Imagine three different experts are answering this question. 
    All experts will write down 1 step
    of their thinking, then share it with the group. Then all experts
    will go on to the next step, etc. If any expert realizes they're
    wrong at any point then they leave. The question is 'The
    cafeteria had 23 apples. If they used 20 to make lunch and bought
    6 more, how many apples do they have?' Make sure to discuss the
    results.
</pre>

Output verification (Structured output and self-critique method)

Constrained sampling (limit output tokens in case of sentiment analysis)

### Processing Unstructured data

In [ ]:
import fitz

In [ ]:
doc = fitz.open("dataset/largedictionary.pdf")

In [ ]:
for page_num, page in enumerate(doc):
    print(f"--- Extracting page {page_num+1} ---")

    text = page.get_text().strip()  # look for embedded text
    images = page.get_images(full=True) # look for scanned image
    
    print("Extracted text length:", len(text))
    print("Number of images:", len(images))

In [ ]:
import traceback
import sys

"""
PDF TEXT EXTRACTION PIPELINE (Teaching Version)

This module demonstrates a production-grade approach to extracting text
from PDFs using a hybrid strategy:

1. Try native text extraction using PyMuPDF (fast, accurate).
2. If no text exists (scanned PDF), fall back to OCR using Tesseract.
3. Handle difficult cases by rendering the entire page as an image.

Key Teaching Concepts:
- PDF ≠ Text
- OCR should be a fallback, not the default
- Layout and resolution matter a LOT for OCR accuracy
"""

# -------------------------------
# Imports
# -------------------------------

import fitz  # PyMuPDF: for reading PDFs and rendering pages
import io    # Used to treat raw image bytes as file-like objects

from typing import List

from PIL import Image      # Pillow: image processing (required for OCR)
import pytesseract         # Python wrapper around Tesseract OCR engine


# -------------------------------
# Helper Function: OCR on Image Bytes
# -------------------------------

def _ocr_image_bytes(
    image_bytes: bytes,
    lang: str = "eng",
    psm: int = 6,
    oem: int = 3,
    dpi: int = 300,
) -> str:
    """
    Perform OCR on a single image represented as raw bytes.

    WHY THIS FUNCTION EXISTS:
    - Keeps OCR logic isolated and reusable
    - Makes the main PDF extraction function easier to read
    - Allows experimentation with OCR parameters independently

    PARAMETERS:
    - image_bytes : Raw bytes of an image extracted from a PDF
    - lang        : OCR language (default: English)
    - psm         : Page Segmentation Mode
                    6 → Assume a single uniform block of text
                    3 → Fully automatic layout detection (best for dictionaries)
    - oem         : OCR Engine Mode
                    3 → Default LSTM-based engine (recommended)
    - dpi         : DPI hint for OCR (higher DPI improves accuracy)

    RETURNS:
    - Extracted text as a string
    """

    # Convert raw bytes into a PIL Image object
    # Tesseract cannot work directly on raw bytes
    img = Image.open(io.BytesIO(image_bytes))

    # Convert image to grayscale
    # This reduces noise and improves OCR accuracy
    img = img.convert("L")

    # Build Tesseract configuration string
    # --oem controls the OCR engine
    # --psm controls how Tesseract segments the page
    config = f"--oem {oem} --psm {psm}"

    # Perform OCR and clean up whitespace
    text = pytesseract.image_to_string(
        img,
        lang=lang,
        config=config
    )

    return text.strip()


# -------------------------------
# Main Function: PDF Text Extraction
# -------------------------------

def extract_pdf_text(
    pdf_path: str,
    ocr_lang: str = "eng",
    ocr_psm: int = 6,
    ocr_oem: int = 3,
    use_page_render_fallback: bool = True,
    render_dpi: int = 300,
    pages_to_process=None,
) -> str:
    """
    Extract text from a PDF using a hybrid strategy.

    HIGH-LEVEL LOGIC:
    -----------------
    For each page in the PDF:
    1. Try native text extraction (PyMuPDF).
    2. If text exists → use it (fast + accurate).
    3. If text does NOT exist → assume scanned page.
    4. Extract images → OCR them.
    5. If that fails → render full page → OCR.

    WHY THIS APPROACH IS IMPORTANT:
    - OCR is slow and error-prone
    - Native text extraction is always preferred
    - This mirrors real-world enterprise pipelines

    RETURNS:
    - A single string containing text from all pages
    """

    # Open the PDF document
    doc = fitz.open(pdf_path)

    # This list will store extracted text page by page
    all_text = dict()

    # Iterate through each page in the PDF
    for page_index, page in enumerate(doc):
        if pages_to_process is not None and page_index == pages_to_process:
            break
        # ------------------------------------
        # STEP 1: Native text extraction
        # ------------------------------------

        # Extract text using PyMuPDF
        # sort=True helps maintain reading order in multi-column layouts
        text = page.get_text("text", sort=True).strip()

        # If text is found, this is a digital (non-scanned) page
        if text:
            all_text[page_index] = text
            continue  # Skip OCR entirely for this page

        # ------------------------------------
        # STEP 2: OCR fallback (page likely scanned)
        # ------------------------------------

        # Collect OCR output for this page
        ocr_text_parts: List[str] = []

        # Extract embedded images from the page
        images = page.get_images(full=True)

        # OCR each embedded image (if any)
        for img in images:
            # xref is an internal PDF reference ID for the image
            xref = img[0]

            try:
                # Extract image bytes from the PDF
                base_image = doc.extract_image(xref)
                image_bytes = base_image["image"]

                # Run OCR on the extracted image
                ocr_text = _ocr_image_bytes(
                    image_bytes=image_bytes,
                    lang=ocr_lang,
                    psm=ocr_psm,
                    oem=ocr_oem,
                )

                # Store OCR text if anything meaningful was found
                if ocr_text:
                    ocr_text_parts.append(ocr_text)

            except Exception:
                # OCR errors should NOT break the pipeline
                # Real-world PDFs often contain malformed images
                continue

        # ------------------------------------
        # STEP 3: Full-page render fallback
        # ------------------------------------

        # If no text was obtained from embedded images,
        # render the entire page as a high-resolution image
        if use_page_render_fallback and not "\n".join(ocr_text_parts).strip():

            try:
                # PDFs are vector-based; OCR needs raster images
                # Convert page to high-resolution bitmap
                zoom = render_dpi / 72  # 72 DPI is PDF default
                matrix = fitz.Matrix(zoom, zoom)

                # Render page to an image
                pix = page.get_pixmap(matrix=matrix, alpha=False)

                # Convert rendered page to PNG bytes
                image_bytes = pix.tobytes("png")

                # OCR the rendered page
                rendered_ocr = _ocr_image_bytes(
                    image_bytes=image_bytes,
                    lang=ocr_lang,
                    psm=ocr_psm,
                    oem=ocr_oem,
                )

                if rendered_ocr:
                    ocr_text_parts.append(rendered_ocr)

            except Exception:
                traceback.print_exc(file=sys.stdout)
                # Rendering or OCR can fail for some pages
                pass

        # ------------------------------------
        # STEP 4: Final page text assembly
        # ------------------------------------

        # Combine all OCR results for the page
        page_text = "\n".join(
            t for t in ocr_text_parts if t.strip()
        ).strip()

        if not page_text:
            page_text = f"NO TEXT EXTRACTED - page {page_index}"
        
        all_text[page_index] = page_text
        
    # Close the PDF to free resources
    doc.close()

    return all_text

In [ ]:
article_on_India = extract_pdf_text(
    "dataset/India.pdf",
    ocr_psm=3  # better for multi-column dictionary pages
)

In [ ]:
article_on_India

In [ ]:
dictionary = extract_pdf_text(
    "dataset/largedictionary.pdf",
    ocr_psm=3,
    pages_to_process=50
)

In [ ]:
dictionary

In [ ]:
india_data_list = []
for key,value in article_on_India.items():
    india_data_list.append({key:value})

In [ ]:
india_data_list

In [ ]:
india_data_list[0:2]

In [ ]:
india_data_str = ''
for item in india_data_list[0:2]:
    # print(list(item.items()))
    key,value = list(item.items())[0]
    india_data_str = india_data_str + f"-----Page {key}-------\n" + f"{value}" + '\n' + '=' * 50 + '\n'

In [ ]:
print(india_data_str)

In [ ]:
prompt_structure = {
    'role': """
        You are an article summarizer.
    """,
    'context': """
        Data is an article on the country India.
    """,
    'data': f"""
        {india_data_str}
    """,
    'task': """
        Summarize the text in each page into understandable content. Make sure the summary is concise and clear. Make the points bulletted
    """,
    'output_format': """
        Page number :
        Summary :
    """,
    'guardrails' : """
        Do not cook up information but rather stay grounded on the content given as data to you.
        Do not cook up the page numbers but look it from the key in the data.
    """,
}

In [ ]:
# https://huggingface.co/Qwen/Qwen2.5-7B-Instruct-GGUF/blob/main/README.md

response = generate_response(
    tokenizer=tokenizer, 
    model=model,
    prompt=get_prompt(prompt_structure=prompt_structure),
    max_new_tokens=8000,
    temperature=0.1,
    top_p=0.1
)
print(response)